In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

In [11]:
import numpy as np
import pandas as pd

np.random.seed(42)

def probability_to_credit_score(p_default):
    # invert because high default risk → low score
    p_good = 1 - p_default
    
    score = 200 + p_good * 600
    return np.clip(score, 200, 800)

def generate_credit_advance_dataset(n=10000):

    # -----------------------------
    # LATENT CUSTOMER QUALITY
    # -----------------------------
    # higher = more financially stable
    #creditworthiness = np.random.beta(a=2, b=2, size=n)
    creditworthiness = np.random.beta(a=10, b=2, size=10000)

    df = pd.DataFrame({
        "creditworthiness": creditworthiness
    })

    # -----------------------------
    # VOICE USAGE (proxy stability)
    # -----------------------------
    df["outgoing_call_count"] = np.random.poisson(5 + 40 * creditworthiness)
    df["incoming_call_count"] = np.random.poisson(5 + 45 * creditworthiness)

    df["outgoing_minutes"] = df["outgoing_call_count"] * np.random.uniform(1, 4, n)
    df["incoming_minutes"] = df["incoming_call_count"] * np.random.uniform(1, 4, n)

    df["avg_call_duration"] = (
        (df["outgoing_minutes"] + df["incoming_minutes"]) /
        (df["outgoing_call_count"] + df["incoming_call_count"] + 1)
    )

    df["outgoing_incoming_ratio"] = (
        df["outgoing_call_count"] /
        (df["incoming_call_count"] + 1)
    )

    # -----------------------------
    # SMS USAGE (Zero-Inflated Poisson)
    # -----------------------------
    
    # probability of being an "inactive SMS user"
    p_zero_sent = 0.4 - 0.3 * creditworthiness   # better users less likely to be zero
    p_zero_sent = np.clip(p_zero_sent, 0.05, 0.6)
    
    is_zero_sent = np.random.binomial(1, p_zero_sent, size=len(df))
    
    lambda_sent = 2 + 20 * creditworthiness
    sms_sent = np.where(
        is_zero_sent == 1,
        0,
        np.random.poisson(lambda_sent)
    )
    
    df["sms_sent"] = sms_sent

    p_zero_received = 0.3 - 0.25 * creditworthiness
    p_zero_received = np.clip(p_zero_received, 0.05, 0.5)
    
    is_zero_received = np.random.binomial(1, p_zero_received, size=len(df))
    
    lambda_received = 3 + 25 * creditworthiness
    sms_received = np.where(
        is_zero_received == 1,
        0,
        np.random.poisson(lambda_received)
    )
    
    df["sms_received"] = sms_received
    
    # -----------------------------
    # DATA USAGE
    # -----------------------------
    df["download_mb"] = np.random.lognormal(
        mean=5 + 2 * creditworthiness,
        sigma=0.5
    )
    df["upload_mb"] = df["download_mb"] * np.random.uniform(0.1, 0.4, n)
    df["avg_daily_data_mb"] = df["download_mb"] / np.random.uniform(20, 30, n)

    # -----------------------------
    # RECHARGE BEHAVIOR (CRITICAL DRIVER)
    # -----------------------------
    df["recharge_count"] = np.random.poisson(1 + 4 * creditworthiness)

    recharge_base = np.random.choice([1, 5, 10], size=n, p=[0.5, 0.3, 0.2])
    df["avg_recharge_amount"] = recharge_base * (1 + creditworthiness)

    df["recharge_amount"] = df["recharge_count"] * df["avg_recharge_amount"]

    # -----------------------------
    # ACTIVE USAGE BEHAVIOR
    # -----------------------------
    df["active_days"] = (10 + 20 * creditworthiness).astype(int)

    df["days_since_last_recharge"] = np.random.exponential(
        scale=20 * (1 - creditworthiness + 0.2)
    )

    df["num_cell_site_latch_ons"] = np.random.poisson(50 + 200 * creditworthiness)

    # -----------------------------
    # CORE BUSINESS MODEL:
    # P(recharge within 15 days)
    # -----------------------------
    # higher creditworthiness → more likely to repay
    logit = (
        -2.0
        + 5.0 * creditworthiness
        + 0.01 * df["active_days"]
        + 0.002 * df["recharge_count"]
        - 0.03 * df["days_since_last_recharge"]
        + 0.001 * df["sms_received"]
    )

    prob_repay_15d = 1 / (1 + np.exp(-logit))

    df["repaid_within_15d"] = np.random.binomial(1, prob_repay_15d)

    # -----------------------------
    # TARGET DEFINITION
    # -----------------------------
    # default = NO repayment within 15 days
    df["default"] = 1 - df["repaid_within_15d"]

    df.drop(columns=["creditworthiness", "repaid_within_15d"], inplace=True)

    return df

In [12]:
df = generate_credit_advance_dataset(10000)

In [13]:
df.columns

Index(['outgoing_call_count', 'incoming_call_count', 'outgoing_minutes',
       'incoming_minutes', 'avg_call_duration', 'outgoing_incoming_ratio',
       'sms_sent', 'sms_received', 'download_mb', 'upload_mb',
       'avg_daily_data_mb', 'recharge_count', 'avg_recharge_amount',
       'recharge_amount', 'active_days', 'days_since_last_recharge',
       'num_cell_site_latch_ons', 'default'],
      dtype='object')

In [14]:
df.head()

,outgoing_call_count,incoming_call_count,outgoing_minutes,incoming_minutes,avg_call_duration,outgoing_incoming_ratio,sms_sent,sms_received,download_mb,upload_mb,avg_daily_data_mb,recharge_count,avg_recharge_amount,recharge_amount,active_days,days_since_last_recharge,num_cell_site_latch_ons,default
0,45,52,129.707396,193.905912,3.302177,0.849057,17,20,950.591440,269.853863,44.040766,5,1.883146,9.415731,27,3.819980,247,0
1,40,40,62.745356,47.500299,1.361057,0.975610,30,0,1305.044810,261.751811,58.029628,5,1.866303,9.331516,27,5.389852,190,0
2,45,43,77.685159,93.521510,1.923670,1.022727,0,25,388.168068,133.958939,14.508927,7,9.217669,64.523682,26,18.783290,215,0
3,38,39,118.318031,72.619986,2.447923,0.950000,15,0,564.383044,216.111019,23.348247,1,8.851149,8.851149,25,1.719085,191,0
4,45,44,136.146864,104.019756,2.668518,1.000000,22,26,393.239432,61.596442,13.709345,2,1.979735,3.959470,29,4.829945,215,0


In [20]:
df.groupby("default").agg({"outgoing_call_count":"count"}).rename(columns={"outgoing_call_count":"count"})

,count
default,
0,8894
1,1106


In [9]:
df.to_csv("../data/synthetic_data.csv",index=False)